# 359. Logger Rate Limiter

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** design, hash-table, sliding-window
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/logger-rate-limiter/)

Design a logger system that receives a stream of messages along with their
timestamps. Each **unique** message should only be printed **at most every 10
seconds** - that is, a message printed at timestamp `t` will not be printed again
until timestamp `t + 10`.

All messages arrive in chronological order, and several messages may arrive at the
same timestamp.

Implement the `Logger` class:

- `Logger()` initialises the object.
- `shouldPrintMessage(timestamp, message)` returns `true` if the message should be
  printed at the given timestamp, otherwise returns `false`.

---

### Example

```
Logger logger = new Logger();
logger.shouldPrintMessage(1,  "foo");   // true,  nothing printed yet
logger.shouldPrintMessage(2,  "bar");   // true,  different message
logger.shouldPrintMessage(3,  "foo");   // false, "foo" printed at 1, next allowed at 11
logger.shouldPrintMessage(8,  "bar");   // false, "bar" printed at 2, next allowed at 12
logger.shouldPrintMessage(10, "foo");   // false, next allowed at 11 - one second early
logger.shouldPrintMessage(11, "foo");   // true,  11 >= 11
```

---

### Constraints

- `0 <= timestamp <= 10^9`
- Every function call is made with a **non-decreasing** value of `timestamp`
- At most `10^4` calls will be made to `shouldPrintMessage`

Every logging library you will ever use has this class in it, usually under the
name "throttle" or "dedupe". It is four lines. The interesting part is the question
LeetCode does not ask you: what happens to the memory after a month.

## Before you write anything

**1.** You need one number per distinct message. There are two things you could
store: **the time it was last printed**, or **the earliest time it may print
again**. Both work. Write the `if` for each. One of them puts a `+ 10` in the
comparison on every single call; the other puts it in one place, once. Pick, and
say why.

**2.** The boundary. Printed at `t = 1`. Is it allowed at `t = 10`? At `t = 11`?
The statement says "will not be printed again until timestamp `t + 10`". Write the
comparison so both of those come out right, then check it against the example -
that `(10, "foo")` line exists precisely to catch a `>` where you needed `>=`.

**3.** Timestamps are **non-decreasing**, not strictly increasing - two messages can
share a timestamp. What does that mean for the *same* message twice at the same
`t`? Work out what your `if` returns for `shouldPrintMessage(1, "foo")` twice in a
row, and confirm it is what you want.

**4.** State changes **only** on the `True` path. Write down what breaks if you
update the dict on every call: give the exact three-call sequence that then returns
the wrong answer. This is the same "mutate only when you succeed" shape as
#1603's `addCar`.

**5.** **The question LeetCode does not ask.** Your dict gets one entry per distinct
message and never removes any. A real server logging a million distinct error
strings holds all million forever, to answer a question about the last 10 seconds.
Sketch a version whose memory is bounded by the number of messages *in the window*
rather than the number ever seen. What do you have to give up to get it?

## Two routes

**A - one dict, message to next-allowed time** *(write this first)*

```
self.next_ok = {}          # message -> the earliest timestamp it may print again
```

`shouldPrintMessage` is: if the message is absent **or** `timestamp` has reached its
next-allowed time, record `timestamp + 10` and return `True`; otherwise return
`False`. `O(1)` per call. `dict.get(message, 0)` collapses the "absent" case into
the comparison and removes an `if` - `0` works as the default precisely because
timestamps start at `0` and can only grow.

That is the accepted answer, and it is four lines.

**B - two buckets, bounded memory** *(question 5)*

Keep **two** dicts: one for the current 10-second window and one for the previous.
Look a message up in both; write only to the current. When `timestamp` crosses into
a new window, drop the old dict and promote the current one. Memory is now
proportional to the messages seen in the last 20 seconds, not to the messages ever
seen - the rest is garbage collected for free when you drop the dict.

The price: a message can now be throttled for up to 20 seconds instead of exactly
10, because the window boundaries are fixed rather than per-message. That is the
trade every production rate limiter makes, and the reason yours is *approximate*.

> **The dict is not the lesson - the retention is.** Route A is the interview
> answer and it is correct for `10^4` calls. Route B is what the same class looks
> like when it has to run for a year, and the difference between them is one
> question nobody asked you. Ask it yourself, every time you write a cache.

In [ ]:
class Logger:

    def __init__(self):
        pass

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        pass

### The test harness

Every call returns a bool, so the answers are visible - but `1 == True` in Python, so
a method that returns something truthy instead of a bool would pass a naive
comparison. The harness type-checks every return.

`check` replays `(timestamp, message)` pairs against your class and against a model
that keeps every message's last-printed time and applies the rule directly. It reports
the first disagreement with the message's history attached, so you can see *why* the
expected answer is what it is rather than just that you got it wrong.

`stress` generates non-decreasing timestamps over a small vocabulary of messages, so
the same string comes back inside, on, and outside its window many times. The seed
makes every run identical. Run this cell; don't edit it.

In [ ]:
import random


def check(calls):
    '''Replay (timestamp, message) pairs against Logger and a direct-rule model.'''
    log = []
    try:
        lg = Logger()
    except Exception as e:
        return False, [f"   !! Logger() raised {type(e).__name__}: {e}"]

    printed = {}                                 # message -> when it last printed
    for ts, msg in calls:
        last = printed.get(msg)
        want = last is None or ts - last >= 10
        try:
            got = lg.shouldPrintMessage(ts, msg)
        except Exception as e:
            log.append(f"   !! shouldPrintMessage({ts}, {msg!r}) raised {type(e).__name__}: {e}")
            return False, log

        log.append(f"shouldPrintMessage({ts:>4}, {msg!r:<8}) -> {got!r}")
        if not isinstance(got, bool):
            log.append(f"   !! must return a bool, got {type(got).__name__} {got!r}")
            log.append(f"      (careful: 1 == True in Python, so a truthy value can look right)")
            return False, log
        if got != want:
            why = ("never printed before" if last is None
                   else f"last printed at {last}, next allowed at {last + 10}")
            log.append(f"   !! must return {want!r}, got {got!r}  -  {msg!r} {why}")
            return False, log
        if want:
            printed[msg] = ts

    return True, log


def stress(n, seed=0, vocab=4, max_step=6):
    '''Non-decreasing timestamps over a small vocabulary, so windows overlap a lot.'''
    random.seed(seed)
    words = [f"msg{i}" for i in range(vocab)]
    t, calls = 0, []
    for _ in range(n):
        calls.append((t, random.choice(words)))
        t += random.randint(0, max_step)         # 0 is legal: same timestamp twice
    return check(calls)


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example",
     [(1, "foo"), (2, "bar"), (3, "foo"), (8, "bar"), (10, "foo"), (11, "foo")]),

    ("question 2: exactly 10 later is ALLOWED",
     [(1, "a"), (11, "a")]),

    ("question 2: 9 later is not, 10 later is",
     [(0, "a"), (9, "a"), (10, "a")]),

    ("question 3: same message twice at the same timestamp",
     [(1, "a"), (1, "a")]),

    ("different messages never block each other",
     [(1, "a"), (1, "b"), (1, "c"), (1, "a")]),

    ("timestamp 0 is legal",
     [(0, "a"), (0, "b"), (0, "a"), (10, "a")]),

    ("question 4: a blocked call must not push the window back",
     [(1, "a"), (5, "a"), (9, "a"), (11, "a")]),

    ("a message that keeps just missing its window",
     [(0, "a"), (9, "a"), (18, "a"), (19, "a"), (29, "a")]),

    ("one message, printed on every tenth second",
     [(t, "tick") for t in range(0, 101, 10)]),

    ("the empty string is a message like any other",
     [(0, ""), (1, ""), (10, "")]),
]

for name, calls in CASES:
    report(name, *check(calls))

for n, seed, vocab, step in [(50, 1, 2, 4), (200, 2, 4, 6), (1000, 3, 8, 3), (10000, 4, 20, 2)]:
    report(f"stress: {n} calls (seed {seed}, {vocab} distinct messages, steps 0..{step})",
           *stress(n, seed, vocab, step))

print("\ntrace of the LeetCode example:")
for line in check([(1, "foo"), (2, "bar"), (3, "foo"),
                   (8, "bar"), (10, "foo"), (11, "foo")])[1]:
    print("  " + line)

## After it passes

- **Measure the leak.** Run 10 000 calls with 10 000 *distinct* messages and print
  `len(self.next_ok)`. Then do it with 4 distinct messages. Route A's memory follows
  the first number, not the second - and the first number is the one that grows
  forever in a real server.
- **Build route B** and run the same tests. Some will fail, and that is not a bug:
  the two-bucket version is deliberately approximate. Work out exactly which case
  disagrees and why, then decide whether you would accept it. Being able to argue
  "this is wrong in a way I chose" is a different skill from being right.
- **The invariant.** *A message returns `True` at time `t` only if it has not
  returned `True` at any time in `(t - 10, t]`.* Write the one line of your method
  that could break it.
- **Make it real.** Real throttles are keyed on a **pair** - `(user_id, endpoint)` -
  not a single string, and they allow *N* per window rather than one. Change the
  value in your dict from a timestamp to a small deque of timestamps and you have
  #933's queue living inside this dict. That combination is, more or less, every
  rate limiter in production.
- Siblings: **#933 Number of Recent Calls** (the same 10-second idea, but counting
  instead of gating - and its queue is what route B's value type becomes),
  #362 Design Hit Counter, #146 LRU Cache (the other answer to "my dict grows
  forever" - bound it by *size* instead of by *time*).